# Fake News Classification using LSTM in PyTorch: End-to-End NLP Pipeline

This notebook implements an end-to-end fake news classification pipeline using PyTorch. It covers text preprocessing, tokenization, vocabulary construction with special tokens, sequence encoding and padding, and building a custom Dataset and DataLoader. An LSTM-based neural network is trained for binary classification, followed by evaluation using accuracy and classification metrics.

The project demonstrates best practices in deep learning-based NLP, including minimal preprocessing, handling unknown tokens, and efficient batching for model training.


Imports


In [58]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import re
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

Load Data

In [59]:


df = pd.read_csv(
    "/content/train.csv",
    engine='python',
    on_bad_lines='skip'
)

df = df[['text', 'class']].dropna()

texts = df['text'].tolist()
labels = df['class'].tolist()

print("Loaded samples:", len(texts))

Loaded samples: 40000


Clean Text

In [13]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

In [60]:
texts = [clean_text(t) for t in texts]

In [61]:
texts[:1]

['that s what we re talking about another campaign promise kept no wonder the democrats and their media allies fear president trump when is the last time a politician actually followed through on a promise they made to the american voters that helped them to get electedpresident trump joined two republican senators on wednesday to champion legislation overhauling legal immigration in america calling for a meritbased system that would significantly cut admissions over the next decadespeaking at the white house the president called it  the most significant reform to our immigration system in a half century as a candidate i campaigned on creating a meritbased immigration system that protects us workers and taxpayers and that is why we are here today  trump saidhe was joined by georgia sen david perdue and arkansas sen tom cotton the republicans who first introduced the reforming american immigration for a strong economy or the raise act in february they have said the legislation aims to r

Tokenization

In [62]:
def tokenize(text):
    return text.split()

tokenized_texts = [tokenize(t) for t in texts]


In [63]:
tokenized_texts[0]

['that',
 's',
 'what',
 'we',
 're',
 'talking',
 'about',
 'another',
 'campaign',
 'promise',
 'kept',
 'no',
 'wonder',
 'the',
 'democrats',
 'and',
 'their',
 'media',
 'allies',
 'fear',
 'president',
 'trump',
 'when',
 'is',
 'the',
 'last',
 'time',
 'a',
 'politician',
 'actually',
 'followed',
 'through',
 'on',
 'a',
 'promise',
 'they',
 'made',
 'to',
 'the',
 'american',
 'voters',
 'that',
 'helped',
 'them',
 'to',
 'get',
 'electedpresident',
 'trump',
 'joined',
 'two',
 'republican',
 'senators',
 'on',
 'wednesday',
 'to',
 'champion',
 'legislation',
 'overhauling',
 'legal',
 'immigration',
 'in',
 'america',
 'calling',
 'for',
 'a',
 'meritbased',
 'system',
 'that',
 'would',
 'significantly',
 'cut',
 'admissions',
 'over',
 'the',
 'next',
 'decadespeaking',
 'at',
 'the',
 'white',
 'house',
 'the',
 'president',
 'called',
 'it',
 'the',
 'most',
 'significant',
 'reform',
 'to',
 'our',
 'immigration',
 'system',
 'in',
 'a',
 'half',
 'century',
 'as',


Build Vocabulary (with PAD & UNK)

In [64]:
def build_vocab(tokenized_texts, max_vocab_size=10000):
    counter = Counter()

    for tokens in tokenized_texts:
        counter.update(tokens)

    vocab = {
        "<PAD>": 0,
        "<UNK>": 1
    }

    most_common = counter.most_common(max_vocab_size - 2)

    for idx, (word, _) in enumerate(most_common, start=2):
        vocab[word] = idx

    return vocab

In [73]:
vocab = build_vocab(tokenized_texts)
vocab_size = len(vocab)
print("Vocabulary size:", vocab_size)

Vocabulary size: 10000


Encode + Pad

In [66]:
def encode(tokens, vocab):
    return [vocab.get(word, vocab["<UNK>"]) for word in tokens]

In [67]:
encoded_texts = [encode(t, vocab) for t in tokenized_texts]
def pad_sequence(seq, max_len):
    if len(seq) < max_len:
        seq += [0] * (max_len - len(seq))
    else:
        seq = seq[:max_len]
    return seq
MAX_LEN = 100

padded_texts = [pad_sequence(seq, MAX_LEN) for seq in encoded_texts]

In [68]:
labels[1]

'Fake'

In [69]:
label_map = {'Fake': 0, 'Real': 1}

labels = [
    label_map[l] if l in label_map else np.random.choice([0, 1])
    for l in labels
]

Train-Test Split

In [71]:
X_train, X_test, y_train, y_test = train_test_split(
    padded_texts, labels, test_size=0.2, random_state=42
)

Custom Dataset

In [74]:
class NewsDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [75]:
train_dataset = NewsDataset(X_train, y_train)
test_dataset = NewsDataset(X_test, y_test)

DataLoader

In [76]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

Model (LSTM)

In [77]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.embedding(x)             # (B, L, E)

        _, (hidden, _) = self.lstm(x)     # hidden → (1, B, H)

        x = hidden.squeeze(0)             # (B, H)

        x = self.fc(x)                   # (B, 1)

        return x

Setup

In [79]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMModel(vocab_size, embed_dim=128, hidden_dim=128).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Training Loop

In [81]:
epochs = 20

for epoch in range(epochs):
    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device).unsqueeze(1)

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 12.6684
Epoch 2, Loss: 7.1584
Epoch 3, Loss: 5.4499
Epoch 4, Loss: 5.2610
Epoch 5, Loss: 4.3369
Epoch 6, Loss: 4.8402
Epoch 7, Loss: 1.2048
Epoch 8, Loss: 1.0365
Epoch 9, Loss: 0.7478
Epoch 10, Loss: 2.3785
Epoch 11, Loss: 2.8882
Epoch 12, Loss: 1.1013
Epoch 13, Loss: 1.0133
Epoch 14, Loss: 0.0557
Epoch 15, Loss: 0.0147
Epoch 16, Loss: 0.0074
Epoch 17, Loss: 0.0042
Epoch 18, Loss: 0.0025
Epoch 19, Loss: 0.0015
Epoch 20, Loss: 0.0009


Evaluation

In [83]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)

        outputs = model(X_batch)
        probs = torch.sigmoid(outputs)

        preds = (probs >= 0.5).int().cpu().numpy()

        all_preds.extend(preds.flatten())
        all_labels.extend(y_batch.numpy())

Metrics

In [84]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(all_labels, all_preds))
print(classification_report(all_labels, all_preds))

Accuracy: 0.9985
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00      4145
         1.0       1.00      1.00      1.00      3855

    accuracy                           1.00      8000
   macro avg       1.00      1.00      1.00      8000
weighted avg       1.00      1.00      1.00      8000

